In [19]:
import torch
from torch.utils.data import Dataset, DataLoader
import tqdm
from src.language_models.dictionary_corpus import Dictionary, Corpus, tokenize
from src.language_models.model import RNNModel as lstm
from src.language_models.utils import move_to_device
import random
import pandas as pd

In [6]:
nounpp = '//scratch2/mrenaudin/colorlessgreenRNNs/NounPP/Stimuli/nounpp.txt'

In [20]:
vocab = Dictionary('/scratch2/mrenaudin/colorlessgreenRNNs/english_data')


In [ ]:
with open(nounpp, "r") as f:
    for line in f:
        line = line.split()
        sentence = line[1:7]
        condition = line[7:9]
        wrong = line[9]
        correct = line[6]
        encoded_sentence = [self.dictionary.word2idx.get(word, self.dictionary.word2idx.get("<unk>")) for word in sentence]


observes
discourages
encourages
avoids
confuses
engages
encourages
discourages
avoids
remembers
understands
greets
observes
admires
encourages
understands
admires
discourages
confuses
inspires
stimulates
criticizes
greets
observes
understands
inspires
understands
criticizes
greets
knows
observes
discourages
knows
remembers
criticizes
engages
inspires
remembers
stimulates
understands
discourages
inspires
stimulates
understands
engages
approves
encourages
criticizes
observes
admires
criticizes
encourages
greets
approves
confuses
knows
observes
confuses
engages
knows
observes
knows
observes
approves
criticizes
remembers
greets
observes
criticizes
encourages
understands
remembers
understands
inspires
observes
avoids
stimulates
confuses
criticizes
knows
understands
encourages
approves
encourages
admires
greets
avoids
discourages
engages
greets
avoids
criticizes
knows
discourages
encourages
observes
avoids
approves
engages
discourages
inspires
observes
engages
understands
encourages
remember

In [134]:
class NounPPDataset(Dataset):
    def __init__(self, nounpp_file, dictionary):
        self.sentences = []
        self.verbs = []
        self.conditions = []
        self.correctness = []
        self.ids = []
        self.encoded_sentences=[]
        self.encoded_verbs = []
        self.dictionary = dictionary

        with open(nounpp_file, "r") as f:
            for line in f:
                line = line.split()
                #sentence = line[:6]  
                sentence = line[:1]
                #verb = line[5]
                verb = line[2]       
                #condition = line[6]  
                condition = line[3]
                #correctness = line[7]
                correctness = line[4]
                #id = int(line[8][2:]) 
                id = int(line[5][2:]) 
                encoded_sentence = [self.dictionary.word2idx.get(word, self.dictionary.word2idx.get("<unk>")) for word in sentence]
                encoded_verb = self.dictionary.word2idx.get(verb, self.dictionary.word2idx.get("<unk>"))

                self.sentences.append(sentence)
                self.verbs.append(verb)
                self.conditions.append(condition)
                self.correctness.append(correctness)
                self.ids.append(id)
                self.encoded_sentences.append(encoded_sentence)
                self.encoded_verbs.append(encoded_verb)

    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, idx):
        return {
            "sentence": self.sentences[idx],
            "encoded_sentence": torch.tensor(self.encoded_sentences[idx], dtype=torch.long),
            "verb": self.verbs[idx],
            "encoded_verb": torch.tensor(self.encoded_verbs[idx], dtype=torch.long),
            "condition": self.conditions[idx],
            "correctness":self.correctness[idx],
            "id": torch.tensor(self.ids[idx], dtype=torch.long),
        }


In [135]:
def evaluate_subject_verb_agreement(model, test_dataloader, device, dictionary, init_sentences):
    correct_predictions = 0
    total_examples = 0
    results = []
    # --- Priming ---
    model.eval()  # Ensure the model is in evaluation mode
    with torch.no_grad():
        priming_hidden = model.init_hidden(1)  # Initialize hidden state for priming
        priming_hidden = (priming_hidden[0].to(device), priming_hidden[1].to(device))
     
        for init_sentence in init_sentences:
            encoded_init_sentence = [dictionary.word2idx.get(word, dictionary.word2idx.get("<unk>")) for word in init_sentence.split()]
            encoded_init_sentence = torch.tensor(encoded_init_sentence, dtype=torch.long).unsqueeze(1).to(device)  # Add batch dimension
            #encoded_init_sentence = encoded_init_sentence.transpose(0,1)
            #print(encoded_init_sentence.shape)
            _, priming_hidden = model(encoded_init_sentence, priming_hidden)  # Update hidden state
        #priming_hidden = (priming_hidden[0].repeat(1,batch_size, 1).to(device), priming_hidden[1].repeat(1,batch_size, 1).to(device))

    # --- End Priming ---
    with torch.no_grad():
        logits_dict = {}
        for batch in test_dataloader:#tqdm.tqdm(test_dataloader, desc="Evaluating"):
            #input_ids = batch["id"].transpose(0,1).to(device)
            sentence = batch['encoded_sentence'].transpose(0,1).to(device)
            verb = batch['encoded_verb'].to(device)
            correctness = batch['correctness']
            id = batch['id']
            conditions = batch['condition']
            sen = batch['sentence']
            
            batch_size = sentence.size(1)
            # hidden = model.init_hidden(batch_size)
            # hidden =(hidden[0].to(device), hidden[1].to(device))
            hidden = (priming_hidden[0].repeat(1, batch_size, 1), priming_hidden[1].repeat(1, batch_size, 1))
            outputs,_ = model(sentence, hidden)
            print('outputs', outputs.shape)
            outputs = outputs.transpose(0,1)  
            print('outputs', outputs.shape)
           
            verb_position = sentence.size(0)-1
            verb_logits = outputs[:,verb_position,:] 
            print('verb_logits', verb_logits.shape)
            
            log_probs = torch.nn.functional.log_softmax(verb_logits, dim=-1)

            for i in range(batch_size):
                sentence_id = id[i].item()
                given_verb_logit = log_probs[i, verb[i]].item()

                results.append({
                    'id': sentence_id,
                    'condition': conditions[i],
                    'correctness': correctness[i],
                    'logit': given_verb_logit
                })

    # Group results by ID and determine correctness
    df = pd.DataFrame(results)
    grouped = df.groupby('id')
    accuracies = {}

    for condition in df['condition'].unique():
        condition_df = df[df['condition'] == condition]
        
        #Regroup by ID
        condition_grouped = condition_df.groupby('id')
        correct_count = 0
        total_count = 0
        
        for name, group in condition_grouped:
            correct_logit = group[group['correctness'] == 'correct']['logit'].iloc[0]
            wrong_logit = group[group['correctness'] == 'wrong']['logit'].iloc[0]
            if correct_logit > wrong_logit:
                correct_count += 1
            total_count +=1    

        accuracies[condition] = correct_count / total_count if total_count >0 else 0.0


    # Create a summary table using pandas
    summary_df = pd.DataFrame(list(accuracies.items()), columns=['Condition', 'Accuracy'])
    print("\nSubject-Verb Agreement Accuracy by Condition:")

    return accuracies

In [148]:
batch_size = 512
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
data_path = "/scratch2/mrenaudin/colorlessgreenRNNs/english_data"  # current directory
dictionary = Dictionary(data_path)
checkpoint_path = "/scratch2/mrenaudin/colorlessgreenRNNs/checkpoints/full_check/epoch_40.pt"  # Replace with your checkpoint path


In [149]:
model = lstm('LSTM', 50001, 200, 650, 2, 0.2, False)
with open(checkpoint_path, 'rb') as f:
    state_dict = torch.load(f, map_location='cuda' if device =='cuda' else 'cpu')
    model.load_state_dict(state_dict)
model.to(device)
model.eval() 

RNNModel(
  (drop): Dropout(p=0.2, inplace=False)
  (encoder): Embedding(50001, 200)
  (rnn): LSTM(200, 650, num_layers=2, dropout=0.2)
  (decoder): Linear(in_features=650, out_features=50001, bias=True)
)

In [150]:
test_dataset = NounPPDataset(nounpp, dictionary)

In [151]:
test_dataloader = DataLoader(test_dataset, batch_size=batch_size)

In [152]:
init_sentences = [
    "In service , the aircraft was operated by a crew of five and could accommodate either 30 paratroopers , 32 <unk> and 28 sitting casualties , or 50 fully equipped troops . <eos>",
    "He even speculated that technical classes might some day be held \" for the better training of workmen in their several crafts and industries . <eos>",
    "After the War of the Holy League in 1537 against the Ottoman Empire , a truce between Venice and the Ottomans was created in 1539 . <eos>",
    "Moore says : \" Tony and I had a good <unk> and off-screen relationship , we are two very different people , but we did share a sense of humour \" . <eos>",
    "<unk> is also the basis for online games sold through licensed lotteries . <eos>"
]


In [153]:
init_sentences = [
    'The aunts behind the chair observe <eos>',
    'The man near the tables admires <eos>',
    'The mother beside the window engages <eos>',
    'The victims beside the chairs approve <eos>',
    'The poets beside the tree engage <eos>'
]

In [154]:
init_sentence = " ".join(["In service , the aircraft was operated by a crew of five and could accommodate either 30 paratroopers , 32 <unk> and 28 sitting casualties , or 50 fully equipped troops . <eos>",
                    "He even speculated that technical classes might some day be held \" for the better training of workmen in their several crafts and industries . <eos>",
                    "After the War of the Holy League in 1537 against the Ottoman Empire , a truce between Venice and the Ottomans was created in 1539 . <eos>",
                    "Moore says : \" Tony and I had a good <unk> and off-screen relationship , we are two very different people , but we did share a sense of humour \" . <eos>",
                    "<unk> is also the basis for online games sold through licensed lotteries . <eos>"])

In [155]:
evaluate_subject_verb_agreement(model, test_dataloader, device, dictionary, init_sentences)

outputs torch.Size([6, 512, 50001])
outputs torch.Size([512, 6, 50001])
verb_logits torch.Size([512, 50001])
outputs torch.Size([6, 512, 50001])
outputs torch.Size([512, 6, 50001])
verb_logits torch.Size([512, 50001])
outputs torch.Size([6, 512, 50001])
outputs torch.Size([512, 6, 50001])
verb_logits torch.Size([512, 50001])
outputs torch.Size([6, 512, 50001])
outputs torch.Size([512, 6, 50001])
verb_logits torch.Size([512, 50001])
outputs torch.Size([6, 512, 50001])
outputs torch.Size([512, 6, 50001])
verb_logits torch.Size([512, 50001])
outputs torch.Size([6, 512, 50001])
outputs torch.Size([512, 6, 50001])
verb_logits torch.Size([512, 50001])
outputs torch.Size([6, 512, 50001])
outputs torch.Size([512, 6, 50001])
verb_logits torch.Size([512, 50001])
outputs torch.Size([6, 512, 50001])
outputs torch.Size([512, 6, 50001])
verb_logits torch.Size([512, 50001])
outputs torch.Size([6, 512, 50001])
outputs torch.Size([512, 6, 50001])
verb_logits torch.Size([512, 50001])
outputs torch.Size(

{'singular_singular': 0.30333333333333334,
 'singular_plural': 0.285,
 'plural_singular': 0.9183333333333333,
 'plural_plural': 0.9216666666666666}

C'est les memes phrases que sur l'autre test set donc ya un truc qui bug dans mon évaluation !!!!

In [117]:
import numpy as np
df = np.load('/scratch2/mrenaudin/colorlessgreenRNNs/nounpp.abl', allow_pickle = True)

In [118]:
df

{'log_p_targets_correct': array([[-10.58398628],
        [-13.61931992],
        [ -9.76531124],
        ...,
        [-12.61817932],
        [-13.27075577],
        [-12.92080021]]),
 'log_p_targets_wrong': array([[-11.72133827],
        [-16.58925629],
        [-14.17847729],
        ...,
        [-14.34117794],
        [-15.27477837],
        [-14.9061842 ]]),
 'score_on_task': 3773,
 'accuracy_score_on_task': 3773,
 'sentences': [['The', 'athlete', 'behind', 'the', 'bike', 'observes'],
  ['The', 'athlete', 'behind', 'the', 'car', 'discourages'],
  ['The', 'athlete', 'behind', 'the', 'car', 'encourages'],
  ['The', 'athlete', 'behind', 'the', 'cat', 'avoids'],
  ['The', 'athlete', 'behind', 'the', 'cat', 'confuses'],
  ['The', 'athlete', 'behind', 'the', 'cat', 'engages'],
  ['The', 'athlete', 'behind', 'the', 'chair', 'encourages'],
  ['The', 'athlete', 'behind', 'the', 'dog', 'discourages'],
  ['The', 'athlete', 'behind', 'the', 'table', 'avoids'],
  ['The', 'athlete', 'behind', '

In [1]:
import pandas as pd
df = pd.read_csv('/scratch2/mrenaudin/colorlessgreenRNNs/NounPP/results.abl')

In [7]:
df['score'] = (df['log_p_targets_correct']>df['log_p_targets_wrong'])

In [2]:
df

,log_p_targets_correct,log_p_targets_wrong,accuracy_score_on_task,p_difference,score_on_task_p_difference,score_on_task_p_difference_std,sentences,nattr,verb_pos,conditions
0,-11.671101,-13.154692,3665,6.600625e-06,0.000042,0.000119,The athlete behind the bike observes,-999,5,singular\tsingular
1,-14.300000,-16.379517,3665,5.390158e-07,0.000042,0.000119,The athlete behind the car discourages,-999,5,singular\tsingular
2,-10.934811,-14.860785,3665,1.747515e-05,0.000042,0.000119,The athlete behind the car encourages,-999,5,singular\tsingular
3,-10.552126,-12.378320,3665,2.192900e-05,0.000042,0.000119,The athlete behind the cat avoids,-999,5,singular\tsingular
4,-11.681628,-14.679475,3665,8.026112e-06,0.000042,0.000119,The athlete behind the cat confuses,-999,5,singular\tsingular
...,...,...,...,...,...,...,...,...,...,...
3995,-10.187896,-11.593899,3665,2.840079e-05,0.000042,0.000119,The women near the trees encourage,-999,5,plural\tplural
3996,-9.332536,-12.499967,3665,8.477077e-05,0.000042,0.000119,The women near the trees remember,-999,5,plural\tplural
3997,-12.969577,-14.157208,3665,1.619590e-06,0.000042,0.000119,The women near the trees stimulate,-999,5,plural\tplural
3998,-13.175953,-14.898318,3665,1.556999e-06,0.000042,0.000119,The women near the trucks inspire,-999,5,plural\tplural


In [13]:
def accuracy_calculation(group):
        correct_count = (group['log_p_targets_correct'] > group['log_p_targets_wrong']).sum()
        total_count = len(group)
        accuracy = correct_count / total_count if total_count > 0 else np.nan
        return pd.Series({'accuracy': accuracy, 'sample_size':total_count})

        result_df = df.groupby('conditions').apply(accuracy_calculation).reset_index()

        return result_df

In [15]:
dff = accuracy_calculation(df)

In [16]:
dff

accuracy          0.91625
sample_size    4000.00000
dtype: float64

In [6]:
for condition in df['conditions'].unique():
    # 2. Compute the number of times log_p_targets_correct > log_p_targets_wrong
    correct_count = ((df['conditions'] == condition) & 
                     (df['log_p_targets_correct'] > df['log_p_targets_wrong'])).sum()
    
    # 3. Divide it by 1000
    result = correct_count / 1000.0
    
    # 4. Print the result
    print(f"Condition: {condition}: {result}")

Condition: singular	singular: 0.941
Condition: singular	plural: 0.565
Condition: plural	singular: 0.528
Condition: plural	plural: 0.872


In [1]:
import pandas as pd 
df = pd.read_csv('/scratch2/mrenaudin/colorlessgreenRNNs/NounPP/results.abl')

In [5]:
df= pd.read_csv('/scratch2/mrenaudin/colorlessgreenRNNs/results_rnn_relu.abl')